# Seoul Bike Sharing Demand — Regression
Predict hourly rented bike demand using weather and calendar features.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor


In [ ]:
df = pd.read_csv('../data/processed/seoul_bike_clean.csv', parse_dates=['date'])
df.head()


## EDA questions
- Which hours have the highest demand?
- How does demand differ by season and operating day?
- Are weather variables useful after controlling for time of day?


In [ ]:
df.describe(include='all').T


In [ ]:
features = ['hour','temperature_c','humidity_pct','wind_speed_ms','visibility_10m','dew_point_temperature_c','solar_radiation_mj_m2','rainfall_mm','snowfall_cm','season','holiday','functioning_day','year','month','day_of_week','is_weekend','hour_sin','hour_cos']
target = 'rented_bike_count'
df = df.sort_values(['date','hour']).reset_index(drop=True)
split = int(len(df)*0.8)
X_train, X_test = df.loc[:split-1, features], df.loc[split:, features]
y_train, y_test = df.loc[:split-1, target], df.loc[split:, target]
categorical = ['season','holiday','functioning_day']
numeric = [c for c in features if c not in categorical]
preprocess = ColumnTransformer([('num', StandardScaler(), numeric), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)])


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42, n_estimators=250, max_depth=3, learning_rate=0.05, loss='huber'),
    'Random Forest': RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1, min_samples_leaf=2, max_features=0.8)
}
results = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', estimator)])
    pipe.fit(X_train, y_train)
    prediction = pipe.predict(X_test)
    results.append({'model': name, 'MAE': mean_absolute_error(y_test, prediction), 'RMSE': mean_squared_error(y_test, prediction, squared=False), 'R2': r2_score(y_test, prediction)})
pd.DataFrame(results).sort_values('RMSE')


## Interpretation
Use the lowest RMSE as the primary selection rule because the target is a count and RMSE penalizes large operational misses. Always report MAE and R² alongside it.
